# Part I — Representing fields and operators

Companion notebook to the Part I slides (`slides/part1.pdf`) and to Part I of the
lecture notes. It follows the slides in order:

1. Tensors and Penrose notation
2. MPS or TT via SVD
3. Historical perspective on TNs
4. Quantics TT: functions and operators
5. Heat equation (explicit time stepping)

Exercises sit right where the concept is introduced. Each one asks for a **prediction
before you run anything**: if you are surprised, you have learnt something. Every
exercise is followed by a **Solution** cell, collapsed by default — try once honestly,
then look rather than get stuck. Stub cells run without error so that the whole
notebook executes top to bottom.

Conventions (shared with `qtade_tn.py`): a tensor train is a list of cores of shape
`(r_left, d, r_right)`; an MPO core has shape `(R_left, d_out, d_in, R_right)`; core 0
carries the **most significant** bit $x_1$ (coarsest scale).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_quimb as qq
import qtade_tn as tn

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})

## 1. Tensors and Penrose notation

### 1.1 Tensors and the curse of dimensionality

A tensor is a multi-dimensional array. An order-6 tensor $T_{n_1\cdots n_6}$ with
$n_j = 1,\dots,10$ already has $10^6$ entries: the number of entries is exponential in
the number of indices. A field on a grid is exactly such an object once each grid index
is split into digits — and real grids are big.

In [ ]:
print(f"order-6 tensor, 10 values per index: {10**6:,d} entries")
for n in (6, 8, 10, 12):
    N = 2 ** n
    print(f"{N:5d}^3 grid: {N**3:>18,d} unknowns   "
          f"({N**3 * 8 / 2**30:>10.1f} GiB per field, float64)")

print("\nDNS needs ~Re^(9/4) degrees of freedom in 3D:")
for Re in (1e3, 1e5, 1e7):
    print(f"  Re = {Re:>8.0e}:  {Re**2.25:>10.2e} dof")

No discretisation fixes this: finite differences, volumes and elements change the
constant, never the exponent. Every classical escape (adaptive meshes, multigrid,
spectral methods, POD) exploits *structure*. Tensor networks are one more of those.

### 1.2 Structured tensors

The slide's example: $f(x,y,z)$ with $x,y,z\in\{0,1\}$ needs 8 numbers. If it happens
to factorise, $f = p(x)\,q(y)\,r(z)$, it needs 6. A tensor network is a systematic way of
finding (approximate) factorisations of this kind for large tensors.

In [ ]:
p, q, r = np.array([1.0, 2.0]), np.array([0.5, -1.0]), np.array([3.0, 1.0])
f = np.einsum("i,j,k->ijk", p, q, r)          # three nodes, no bonds
print("f has", f.size, "entries; the factors have", p.size + q.size + r.size)
print("f(0,1,0) =", f[0, 1, 0], "=", p[0] * q[1] * r[0])

### 1.3 Diagrams are `einsum`

A node is a tensor, a leg an index; **joined legs are summed**, a **closed loop** (no
free legs) is a number. `np.einsum` is the same language written in letters.

In [ ]:
rng = np.random.default_rng(0)
v, w = rng.standard_normal(4), rng.standard_normal(4)
M, A, B = rng.standard_normal((3, 4, 4))
T = rng.standard_normal((4, 4, 4))

print("vector inner product  sum_j v_j w_j   :", np.einsum("j,j->", v, w))
print("matrix-vector         sum_j M_ij v_j  :", np.einsum("ij,j->i", M, v))
print("matrix trace          sum_j M_jj      :", np.einsum("jj->", M))
print("closed loop           A_ij B_ji       :", np.einsum("ij,ji->", A, B),
      "= Tr[AB] =", np.trace(A @ B))
print("order-3 with a vector sum_j T_ijk v_j : shape", np.einsum("ijk,j->ik", T, v).shape)

The diagram fixes *what* is computed, not *in which order*. The order sets the cost:
finding the optimal order is NP-hard for a general network, but obvious for a chain.

---
### Exercise 1.1 — contraction order is cost


Contract $A_{ij}B_{jk}C_{kl}v_l$ for $D\times D$ matrices with $D=400$, once as
$((AB)C)v$ and once as $A(B(Cv))$. **Predict** the cost of each in powers of $D$, then
time both and confirm with `np.einsum_path`.

In [ ]:
D = 400
A, B, C = rng.standard_normal((3, D, D))
v = rng.standard_normal(D)

# TODO: time ((A @ B) @ C) @ v  and  A @ (B @ (C @ v)); check they agree.
# TODO: print np.einsum_path("ij,jk,kl,l->i", A, B, C, v, optimize="optimal")[1]

**Solution** (hidden by default; expand the cell below)

In [ ]:
t0 = time.perf_counter(); left = ((A @ B) @ C) @ v; t_left = time.perf_counter() - t0
t0 = time.perf_counter(); right = A @ (B @ (C @ v)); t_right = time.perf_counter() - t0
assert np.allclose(left, right)
print(f"((AB)C)v : {t_left * 1e3:7.2f} ms   ~ 2 D^3 flops")
print(f"A(B(Cv)) : {t_right * 1e3:7.2f} ms   ~ 3 D^2 flops")
print(np.einsum_path("ij,jk,kl,l->i", A, B, C, v, optimize="optimal")[1])

## 2. MPS or TT via SVD

### 2.1 Low-rank structure of a matrix: the SVD

$M = U S V^\dagger$ with $U^\dagger U = V^\dagger V = \mathbb{1}_r$ and $S$ diagonal,
holding the singular values: real, non-negative, decreasingly ordered. Grouping
$A = U\sqrt S$, $B = \sqrt S V^\dagger$ gives two nodes joined by a bond of dimension
$r = \mathrm{rank}(M)$. **Truncating** the smallest singular values gives the optimal
approximation in the 2-norm and Frobenius norm (Eckart–Young).

The slide's $4\times 3$ example:

In [ ]:
M = np.array([[0.435839, 0.223707, 0.10],
              [0.435839, 0.223707, -0.10],
              [0.223707, 0.435839, 0.10],
              [0.223707, 0.435839, -0.10]])
U, s, Vt = np.linalg.svd(M, full_matrices=False)
print("singular values:", s)
print("U^T U = 1:", np.allclose(U.T @ U, np.eye(3)), "   V^T V = 1:", np.allclose(Vt @ Vt.T, np.eye(3)))

### 2.2 Two compression strategies

* fix a **maximum rank** $\chi$ → the error is $\epsilon = \sqrt{\sum_{j>\chi}\lambda_j^2}$;
* fix a **maximum error** $\epsilon$ → take the smallest $\chi$ with
  $\sqrt{\sum_{j>\chi}\lambda_j^2}\le\epsilon$.

---
### Exercise 2.1 — truncation error is the discarded tail


Write `truncate(M, chi)` (keep the $\chi$ largest singular values) and `chi_for(s, eps)`
(smallest rank whose discarded tail has norm $\le\epsilon$). Check that
$\|M_{\chi=2}-M\|_F^2 = 0.2^2$ and $\|M_{\chi=1}-M\|_F^2 = 0.2^2+0.3^2$.

In [ ]:
def truncate(M, chi):
    # TODO: SVD, keep chi singular triplets, rebuild
    ...


def chi_for(s, eps):
    # TODO: smallest chi with sqrt(sum_{j >= chi} s_j^2) <= eps  (0-based)
    ...

**Solution** (hidden by default; expand the cell below)

In [ ]:
def truncate(M, chi):
    U, s, Vt = np.linalg.svd(M, full_matrices=False)
    return (U[:, :chi] * s[:chi]) @ Vt[:chi]


def chi_for(s, eps):
    tails = [np.sqrt(np.sum(s[k:] ** 2)) for k in range(len(s) + 1)]
    return next(k for k, t in enumerate(tails) if t <= eps)


for chi in (2, 1):
    err2 = np.linalg.norm(truncate(M, chi) - M) ** 2
    print(f"chi={chi}: ||M_chi - M||^2 = {err2:.4f}   tail = {np.sum(s[chi:] ** 2):.4f}")
print("eps=0.25 -> chi =", chi_for(s, 0.25), "   eps=0.1 -> chi =", chi_for(s, 0.1))

### 2.3 A physical field is not an arbitrary vector

The bet behind the whole course, and it is falsifiable. Reshape a smooth field and a
noise field of $2^{12}$ entries into $64\times 64$ matrices and compare the singular
values. **Predict first.**

In [ ]:
n = 12
x = np.linspace(0, 1, 2 ** n, endpoint=False)
smooth = np.exp(-3 * x) + np.sin(4 * np.pi * x)
noise = np.random.default_rng(0).standard_normal(2 ** n)

fig, ax = plt.subplots(1, 2)
for label, vec in [("smooth", smooth), ("noise", noise)]:
    s_cut = np.linalg.svd(vec.reshape(2 ** (n // 2), -1), compute_uv=False)
    ax[0].semilogy(s_cut / s_cut[0], label=label)
ax[0].set(xlabel="singular value index", ylabel="$\\sigma_k/\\sigma_0$",
          title="decay across the middle cut")
ax[0].legend()
ax[1].plot(x, smooth, label="smooth")
ax[1].plot(x, noise, lw=0.4, alpha=0.7, label="noise")
ax[1].set(xlabel="x", title="the two fields")
ax[1].legend()
plt.tight_layout()

Machine precision in a handful of singular values versus no decay at all: that gap
*is* the compression.

### 2.4 Recursive SVD on a tensor: the MPS / tensor train

1. reshape the tensor as a matrix (first index vs the rest);
2. SVD it, keep $U_1$ as the first core;
3. glue $S_1V_1^T$ together and reshape (bond + next index vs the rest);
4. sweep through the tensor.

The result is a **matrix product state** (physics) or **tensor train** (numerical
analysis): storage goes from $d^N$ to $N d\chi^2$.

---
### Exercise 2.2 — write TT-SVD yourself


Implement `my_tt_svd(T, chi_max)` following the four steps above (cores of shape
`(r_left, d, r_right)`). Test it on the `smooth` field reshaped to $[2]^{12}$: the
reconstruction error should be tiny for `chi_max=4`, and your ranks should match
`tn.tt_svd`. Then print the storage $N d\chi^2$ vs $d^N$.

In [ ]:
def my_tt_svd(T, chi_max):
    cores, rank = [], 1
    rest = T.reshape(1, -1)
    # TODO: loop over all but the last index: reshape, SVD, truncate to chi_max,
    #       append U as a core, carry S V^T forward. Append the remainder last.
    ...

**Solution** (hidden by default; expand the cell below)

In [ ]:
def my_tt_svd(T, chi_max):
    dims = T.shape
    cores, rank = [], 1
    rest = T.reshape(1, -1)
    for d in dims[:-1]:
        U, s_, Vt = np.linalg.svd(rest.reshape(rank * d, -1), full_matrices=False)
        keep = min(chi_max, int(np.sum(s_ > 1e-14 * s_[0])))
        cores.append(U[:, :keep].reshape(rank, d, keep))
        rest = s_[:keep, None] * Vt[:keep]
        rank = keep
    cores.append(rest.reshape(rank, dims[-1], 1))
    return cores


T12 = smooth.reshape([2] * 12)
mine = my_tt_svd(T12, chi_max=4)
print("my ranks:      ", tn.tt_ranks(mine))
print("tn.tt_svd ranks:", tn.tt_ranks(tn.tt_svd(T12, eps=1e-12)))
print(f"relative error: {np.linalg.norm(tn.tt_full(mine) - T12) / np.linalg.norm(T12):.2e}")
print(f"storage: {tn.tt_size(mine)} floats (N d chi^2 <= {12 * 2 * 4**2})  vs  d^N = {2**12}")

### 2.5 Primitives

Efficient manipulations of an MPS — never forming the full vector:

1. **sum** — cores stack block-diagonally, $\chi$ is sub-additive;
2. **element-wise product** — $\chi$ multiplies (then round);
3. perfect sampling;
4. linear system solver (Part II);
5. **inner product** — sweep an environment along the chain, cost $\sim\chi^3$.

An **MPO** (TT operator) has two physical legs per core; an MPO–MPS contraction
returns an MPS of bond dimension $\chi\,\chi_{\mathrm{MPO}}$.

In [ ]:
a = tn.qtt_from_vector(np.sin(4 * np.pi * x), eps=1e-12)
b = tn.qtt_from_vector(np.exp(-3 * x), eps=1e-12)
print("chi(a) =", max(tn.tt_ranks(a)), "  chi(b) =", max(tn.tt_ranks(b)))
print("sum     : chi =", max(tn.tt_ranks(tn.tt_add(a, b))), " -> rounded",
      max(tn.tt_ranks(tn.tt_round(tn.tt_add(a, b), eps=1e-12))))
print("product : chi =", max(tn.tt_ranks(tn.tt_hadamard(a, b))), " -> rounded",
      max(tn.tt_ranks(tn.tt_round(tn.tt_hadamard(a, b), eps=1e-12))))
L_demo = tn.qtt_laplacian(n)
print("MPO-MPS : chi =", max(tn.tt_ranks(tn.mpo_apply(L_demo, a))),
      f"(= chi(a) * chi_MPO = {max(tn.tt_ranks(a))} * {max(tn.mpo_ranks(L_demo))})")

---
### Exercise 2.3 — the inner product of two trillion-entry vectors


Implement `my_dot(a, b)`: sweep an environment matrix $E$ (shape $\chi_a\times\chi_b$)
from left to right, contracting one core of each train per step. **Predict** the cost
per core. Then time it on two random trains of $n=40$ cores ($2^{40}\approx 10^{12}$
entries) with $\chi=100$ and compare with `tn.tt_dot`.

In [ ]:
def random_tt(n, chi, seed):
    g = np.random.default_rng(seed)
    ranks = [1] + [chi] * (n - 1) + [1]
    return [g.standard_normal((ranks[k], 2, ranks[k + 1])) / np.sqrt(2 * chi)
            for k in range(n)]


def my_dot(a, b):
    E = np.ones((1, 1))
    # TODO: for each pair of cores, E <- sum_{a,c,i} E[a,c] a[a,i,b] b[c,i,d]
    #       (do it in two einsums so the cost stays chi^3)
    ...

**Solution** (hidden by default; expand the cell below)

In [ ]:
def my_dot(a, b):
    E = np.ones((1, 1))
    for ca, cb in zip(a, b):
        E = np.einsum("ac,aib->cib", E, ca)      # chi^3 d
        E = np.einsum("cib,cid->bd", E, cb)      # chi^3 d
    return float(E.reshape(()))


ta, tb = random_tt(40, 100, 1), random_tt(40, 100, 2)
t0 = time.perf_counter(); val = my_dot(ta, tb); dt_dot = time.perf_counter() - t0
print(f"<a|b> = {val:.6e} in {dt_dot * 1e3:.1f} ms   (tn.tt_dot: {tn.tt_dot(ta, tb):.6e})")

More advanced algorithms, met later in the course: canonicalisation (gauge freedom),
belief-propagation gauging, TCI (build a train by *querying* the function; Part II),
multiscale interpolative construction, time evolution (TEBD, TDVP; Part III).

## 3. Historical perspective on TNs

No code in this section — see the slides and the notes. In one paragraph: MPS emerged in
quantum many-body physics (transfer matrices, Kramers–Wannier 1941; finitely correlated
states, Fannes–Nachtergaele–Werner 1992; DMRG, White 1992, understood as variational
MPS by Östlund–Rommer 1995). Numerical analysis rediscovered the same object as the
**tensor train** (Oseledets–Tyrtyshnikov 2009–2011). Other networks (TTN, PEPS, MERA)
trade computational cost for expressivity; loops break the canonical form. Libraries:
ITensor, TeNPy, TensorNetwork, **quimb** (used in Parts III–IV and in
`01-heat-equation-qtmps-quimb.ipynb`) and cotengra for contraction ordering.

## 4. Quantics TT: functions and operators

### 4.1 Binary indices label scales

$f:[0,1)\to\mathbb R$ on $N=2^n$ points. Write the grid point in binary,

$$x = \sum_{m=1}^{n} 2^{-m} x_m = \tfrac12 x_1 + \tfrac14 x_2 + \dots + \tfrac1{2^n}x_n,
\qquad x_m\in\{0,1\},$$

so the $2^n$ samples become an order-$n$ tensor $f_{x_1\dots x_n}$ (≈ amplitude encoding
$|f\rangle=\sum f_{x_1\dots x_n}|x_1\rangle\otimes\dots\otimes|x_n\rangle$). $x_1$ is
the coarsest scale, $x_n$ the finest. Low rank $\Leftrightarrow$ low inter-scale
correlations.

In [ ]:
n = 3
xs3 = np.arange(2 ** n) / 2 ** n
f3 = tn.qtt_from_vector(xs3, eps=1e-14)                 # f(x) = x on 8 points
bits = (0, 1, 1)
print("x(0,1,1) =", sum(b / 2 ** (m + 1) for m, b in enumerate(bits)))
print("f(0,1,1) by contracting the cores:", qq.evaluate(f3, bits))

### 4.2 Rank tracks smoothness, not resolution

Doubling the resolution should add one core and leave the bond dimensions alone. The
mesh size scales exponentially in $n$, the memory linearly.

In [ ]:
print(f"{'n':>4} {'N':>9} {'max chi':>8} {'params':>9} {'ratio':>10}")
for n in range(6, 21, 2):
    xs = np.linspace(0, 1, 2 ** n, endpoint=False)
    c = tn.qtt_from_vector(np.exp(-3 * xs) + np.sin(4 * np.pi * xs), eps=1e-10)
    print(f"{n:>4} {2**n:>9,d} {max(tn.tt_ranks(c)):>8} {tn.tt_size(c):>9,d} "
          f"{2**n / tn.tt_size(c):>9.0f}x")

### 4.3 Analytical construction of 1D functions

$$e^{x} = e^{x_1/2 + x_2/4 + \dots + x_n/2^n}
= \begin{pmatrix}1\\ e^{1/2}\end{pmatrix}\otimes\begin{pmatrix}1\\ e^{1/4}\end{pmatrix}
\otimes\dots\otimes\begin{pmatrix}1\\ e^{1/2^n}\end{pmatrix}
\qquad(\text{rank }1),$$

$\sin x,\cos x$ are rank 2 (sums of two exponentials), a monomial $x^p$ is rank $p+1$.
For $f(x)=x$ the slides build the vector $(1,2,\dots,2^n)$ recursively,

$$\begin{pmatrix}1\\ \vdots\\ 2^n\end{pmatrix} = \begin{pmatrix}1\\0\end{pmatrix}\otimes[n-1]
+ \begin{pmatrix}0\\1\end{pmatrix}\otimes\Big([n-1] + 2^{n-1}\begin{pmatrix}1\\1\end{pmatrix}^{\otimes(n-1)}\Big),$$

which carries two states across every bond: $\chi=2$.

---
### Exercise 4.1 — build exp(x) and x by hand


Without any SVD, write down the cores of

* `exp_cores(n)`: $e^{x}$ on $x = j/2^n$, rank 1 — core $m$ is $(1, e^{2^{-m}})$;
* `lin_cores(n)`: $x = \sum_m 2^{-m}x_m$, rank 2. Hint: bond state 0 = "constant 1",
  state 1 = "accumulated value"; core $m$ maps $(1, s)\mapsto(1, s + 2^{-m}x_m)$.

Check both against the dense vectors for $n=10$.

In [ ]:
def exp_cores(n):
    # TODO: list of n arrays of shape (1, 2, 1)
    ...


def lin_cores(n):
    # TODO: first core (1, 2, 2), middle (2, 2, 2), last (2, 2, 1)
    ...

**Solution** (hidden by default; expand the cell below)

In [ ]:
def exp_cores(n):
    return [np.array([1.0, np.exp(2.0 ** -m)]).reshape(1, 2, 1) for m in range(1, n + 1)]


def lin_cores(n):
    cores = []
    for m in range(1, n + 1):
        c = np.zeros((2, 2, 2))
        c[0, :, 0] = 1.0                       # carry the constant 1
        c[1, :, 1] = 1.0                       # pass the accumulated value on
        c[0, 1, 1] = 2.0 ** -m                 # digit x_m adds 2^-m
        cores.append(c)
    cores[0] = cores[0][0:1]                   # start in state "constant"
    cores[-1] = cores[-1][:, :, 1:2]           # end by reading the value
    return cores


n = 10
xs = np.arange(2 ** n) / 2 ** n
for name, cores, ref in [("exp(x)", exp_cores(n), np.exp(xs)), ("x", lin_cores(n), xs)]:
    err = np.max(np.abs(tn.qtt_to_vector(cores) - ref))
    print(f"{name:>6}: ranks {tn.tt_ranks(cores)}   max error {err:.1e}")

---
### Exercise 4.2 — predict the rank


**Point:** the quantics bond dimension measures how many scales are *coupled*, not how
many grid points there are. For each function below write down your guess for the
maximum bond dimension at $\epsilon=10^{-8}$, then measure with
`max(tn.tt_ranks(tn.qtt_from_vector(f, eps=1e-8)))`. At least one will not do what you
expect.

In [ ]:
n = 14
x = np.linspace(0, 1, 2 ** n, endpoint=False)
candidates = {
    "1": np.ones_like(x),
    "x**2": x ** 2,
    "exp(5x) + exp(-3x)": np.exp(5 * x) + np.exp(-3 * x),
    "sin(2 pi x) * exp(-x)": np.sin(2 * np.pi * x) * np.exp(-x),
    "sin(2 pi x) + sin(64 pi x)": np.sin(2 * np.pi * x) + np.sin(64 * np.pi * x),
    "gaussian, width 0.05": np.exp(-((x - 0.5) / 0.05) ** 2),
    "step at x = 1/3": (x > 1 / 3).astype(float),
    "sin(1/(x + 0.01))": np.sin(1 / (x + 0.01)),
    "white noise": np.random.default_rng(1).standard_normal(2 ** n),
}
# TODO: print the measured chi for each candidate

**Solution** (hidden by default; expand the cell below)

In [ ]:
for name, f in candidates.items():
    print(f"{name:>28}: chi = {max(tn.tt_ranks(tn.qtt_from_vector(f, eps=1e-8)))}")
print('''
* exp(5x)+exp(-3x): rank 2, a sum of m exponentials has rank <= m.
* sin(2 pi x)+sin(64 pi x): each sine is rank 2, so the sum is <= 4, fine oscillation or not.
* step: rank 2. Quantics does not see a jump, it sees a comparison 1[i > m], decided
  digit by digit by a two-state automaton ("still equal so far" / "already decided").
* sin(1/(x+0.01)): an oscillation the coarse cores cannot resolve couples many scales.
* noise: no structure, the rank saturates at the maximum allowed by the cut.''')

### 4.4 What functional class?

Tree tensor networks (QTT included) achieve optimal approximation rates for smooth
functions (Sobolev/Besov) of any smoothness order (Ali & Nouy), with constructive
proofs and fast compression (Lindsey's multiscale interpolative construction), rank
bounds for band-limited functions and for functions with cusps or discontinuities.

**But** the converse fails: a *random* MPS with $\chi=2$ is not a nice function.

In [ ]:
g = np.random.default_rng(7)
n = 10
rand_cores = [g.standard_normal((1 if k == 0 else 2, 2, 1 if k == n - 1 else 2))
              for k in range(n)]
plt.plot(tn.qtt_to_vector(rand_cores), lw=0.8)
plt.title("a random QTT with $\\chi=2$: low rank does not mean smooth")
plt.xlabel("grid index")
plt.tight_layout()

---
### Exercise 4.3 — where does rank actually come from?


**Point:** the enemy is scale coupling, not sharpness. Sweep the width $w$ of a Gaussian
bump $e^{-((x-1/2)/w)^2}$ from $10^{-4}$ (narrower than one grid cell at $n=12$) to
$10^{-0.5}$ and plot $\chi$ against $w$. Predict the shape first: monotone? peaked?
flat? Then explain both ends.

In [ ]:
n = 12
x = np.linspace(0, 1, 2 ** n, endpoint=False)
widths = np.logspace(-4, -0.5, 25)
# TODO: chis = [...]; plt.semilogx(widths, chis, "o-")

**Solution** (hidden by default; expand the cell below)

In [ ]:
chis = [max(tn.tt_ranks(tn.qtt_from_vector(np.exp(-((x - 0.5) / w) ** 2), eps=1e-8)))
        for w in widths]
plt.semilogx(widths, chis, "o-")
plt.axvline(2.0 ** -n, ls="--", c="k", lw=1)
plt.text(2.0 ** -n * 1.2, max(chis) * 0.9, "one grid cell", fontsize=9)
plt.xlabel("Gaussian width"); plt.ylabel("max bond dimension")
plt.title("rank peaks where the feature spans several, but not all, scales")
plt.tight_layout()
print("Wide bump: smooth, few scales involved, cheap.")
print("Sub-cell bump: a single spike = a product of one-hot vectors per digit, cheap again.")

### 4.5 Higher dimensions: bit ordering

In 2D/3D (and space + time) we choose how to order the bits along the chain:

* **stacked** — $x_1\dots x_n\,y_1\dots y_n$;
* **interleaved** (Morton / Z-ordering) — $x_1y_1x_2y_2\dots$, one scale at a time;
* **concatenated** — an extra leading index $p\mapsto\{u,v,w\}$ for the field components.

A straight edge is free; a curved one is not, because a circle couples every length
scale at once.

In [ ]:
for m in (6, 8, 10):
    N = 2 ** m
    xx = np.linspace(0, 1, N, endpoint=False)
    XX, YY = np.meshgrid(xx, xx, indexing="ij")
    half_plane = (XX > 0.3).astype(float)
    disc = (((XX - 0.5) ** 2 + (YY - 0.5) ** 2) < 0.25 ** 2).astype(float)
    print(f"N = {N:>5}:  half-plane chi = "
          f"{max(tn.tt_ranks(qq.from_grid(half_plane, eps=1e-8))):>4},   "
          f"disc chi = {max(tn.tt_ranks(qq.from_grid(disc, eps=1e-8))):>4}")

---
### Exercise 4.4 — stacked versus interleaved


**Point:** the ordering is a modelling decision, and which one wins depends on the
field. `qq.from_grid` takes `ordering="interleaved"` or `"stacked"`. Compare them on

* a **separable** field $e^{-40(x-\frac12)^2}e^{-40(y-\frac12)^2}$,
* a **curved sharp** field (disc indicator),
* a **multiscale** field (synthetic $k^{-5/3}$ turbulence),

twice: at matched *tolerance* (`eps=1e-8`, report $\chi$ and parameters) and at matched
*bond dimension* (`chi_max=32`, report the relative $L_2$ error; use `qq.to_grid` for
interleaved and `tn.tt_full(...).reshape(N, N)` for stacked). Predict the winners.

In [ ]:
m = 8
N = 2 ** m
xx = np.linspace(0, 1, N, endpoint=False)
X, Y = np.meshgrid(xx, xx, indexing="ij")
R = np.hypot(X - 0.5, Y - 0.5)

rng3 = np.random.default_rng(3)
kf = np.fft.fftfreq(N) * N
KX, KY = np.meshgrid(kf, kf, indexing="ij")
Kmag = np.hypot(KX, KY)
Kmag[0, 0] = 1.0
spec = Kmag ** (-5 / 6)
spec[Kmag > N / 3] = 0.0
phase = rng3.standard_normal((N, N)) + 1j * rng3.standard_normal((N, N))
turbulent = np.real(np.fft.ifft2(spec * phase))
turbulent /= turbulent.std()

fields = {
    "separable gaussian": np.exp(-40 * (X - 0.5) ** 2) * np.exp(-40 * (Y - 0.5) ** 2),
    "disc indicator": (R < 0.25).astype(float),
    "k^-5/3 turbulence": turbulent,
}
# TODO (a): matched tolerance;  TODO (b): matched bond dimension

**Solution** (hidden by default; expand the cell below)

In [ ]:
print("(a) matched tolerance, eps = 1e-8")
for name, f in fields.items():
    ci = qq.from_grid(f, eps=1e-8, ordering="interleaved")
    cs = qq.from_grid(f, eps=1e-8, ordering="stacked")
    print(f"  {name:>20}:  interleaved chi={max(tn.tt_ranks(ci)):>4} "
          f"({tn.tt_size(ci):>7,d} params)   "
          f"stacked chi={max(tn.tt_ranks(cs)):>4} ({tn.tt_size(cs):>7,d} params)")

print("\n(b) matched bond dimension, chi_max = 32")
for name, f in fields.items():
    ci = qq.from_grid(f, eps=1e-14, chi_max=32, ordering="interleaved")
    cs = qq.from_grid(f, eps=1e-14, chi_max=32, ordering="stacked")
    ei = np.linalg.norm(qq.to_grid(ci) - f) / np.linalg.norm(f)
    es = np.linalg.norm(tn.tt_full(cs).reshape(N, N) - f) / np.linalg.norm(f)
    print(f"  {name:>20}:  interleaved L2={ei:.4f}   stacked L2={es:.4f}")

print('''
* A product f(x)g(y) has rank exactly 1 across the stacked x|y cut, so stacking is
  optimal for anything separable (and many textbook test functions are separable).
* Matched tolerance measures how many numbers each ordering needs to be exact; matched
  bond dimension measures how good it is at a budget you already fixed. For multiscale
  fields they disagree.
* In a large simulation you choose chi (memory), not the tolerance: (b) is the one
  that matters. Pisoni et al. (2026) ask the same question in 3D turbulence.''')

### 4.6 Operators as MPOs

A shift by one grid point is *adding 1 to a binary number*: a carry automaton with two
states, hence $\chi_{\mathrm{MPO}}=2$. The three-point Laplacian is a sum of three shifts
and rounds to $\chi_{\mathrm{MPO}}=3$ (Kazeev & Khoromskij). Nothing is fitted, and the
dense $2^n\times 2^n$ matrix is never formed.

In [ ]:
n = 4
S = tn.qtt_shift(n, +1)
print("shift MPO ranks:", tn.mpo_ranks(S))
print(np.round(tn.mpo_full(S)).astype(int)[:6, :6], "  <- ones on the subdiagonal\n")

L = tn.qtt_laplacian(n, dx=1.0)
print("Laplacian MPO ranks:", tn.mpo_ranks(L), " (2 + 2 + 1 = 5 before rounding)")
print(np.round(tn.mpo_full(L)).astype(int)[:5, :5])

The slides build the (positive) Laplacian $\Delta_{2^n}=\mathrm{tridiag}(-1,2,-1)$
explicitly. With $\mathbb 1_2$, $J=\begin{pmatrix}0&1\\0&0\end{pmatrix}$ and
$\Delta_2 = 2\cdot\mathbb 1_2 - (J+J^T)$:

$$\Delta_4 = \mathbb 1_2\otimes\Delta_2 - (J\otimes J^T + J^T\otimes J)
= \begin{pmatrix}\mathbb 1_2 & -J & -J^T\end{pmatrix}\bowtie
\begin{pmatrix}\Delta_2\\ J^T\\ J\end{pmatrix},$$

$$\Delta_{2^n} = \begin{pmatrix}\mathbb 1_2 & -J & -J^T\end{pmatrix}\bowtie
\begin{pmatrix}\mathbb 1_2 & -J & -J^T\\ 0 & J^T & 0\\ 0 & 0 & J\end{pmatrix}^{\bowtie(n-2)}
\bowtie\begin{pmatrix}\Delta_2\\ J^T\\ J\end{pmatrix},$$

where $\bowtie$ is the matrix product over the bond with Kronecker products of the
$2\times 2$ entries ("each entry is now a $2\times2$ matrix"). Bond dimension 3.

---
### Exercise 4.5 — the Laplacian MPO from the slide


Build `delta_mpo(n)` from the three cores above (MPO core shape
`(R_left, d_out, d_in, R_right)`, first core = coarsest bit) and check that
`tn.mpo_full(delta_mpo(n))` equals `-tn.mpo_full(tn.qtt_laplacian(n, dx=1.0))` for
$n=2,\dots,6$.

In [ ]:
I2 = np.eye(2)
J = np.array([[0.0, 1.0], [0.0, 0.0]])
D2 = 2 * I2 - (J + J.T)


def delta_mpo(n):
    # TODO: first core (1,2,2,3), n-2 middle cores (3,2,2,3), last core (3,2,2,1)
    ...

**Solution** (hidden by default; expand the cell below)

In [ ]:
def delta_mpo(n):
    Z = np.zeros((2, 2))
    first = np.stack([I2, -J, -J.T], axis=-1)[None]                   # (1,2,2,3)
    blocks = [[I2, -J, -J.T], [Z, J.T, Z], [Z, Z, J]]
    mid = np.array([[blocks[a][b] for b in range(3)] for a in range(3)])  # (3,3,2,2)
    mid = mid.transpose(0, 2, 3, 1)                                    # (3,2,2,3)
    last = np.stack([D2, J.T, J], axis=0)[..., None]                   # (3,2,2,1)
    return [first] + [mid] * (n - 2) + [last]


for n in range(2, 7):
    assert np.allclose(tn.mpo_full(delta_mpo(n)), -tn.mpo_full(tn.qtt_laplacian(n, dx=1.0)))
print("slide construction matches tn.qtt_laplacian; ranks:", tn.mpo_ranks(delta_mpo(6)))

---
### Exercise 4.6 — boundary conditions live in the edge vectors


`tn.qtt_shift` builds the non-periodic shift by forcing the left carry bond to state 0
("no overflow off the top"). Allow the overflow to wrap around and you get the
**periodic** shift (cyclic permutation). Change only the edge cores of the carry
automaton below so that `tn.mpo_full(P)` equals `np.roll(np.eye(2**n), 1, axis=0)`.

In [ ]:
def qtt_shift_periodic(n):
    core = np.zeros((2, 2, 2, 2))            # [carry_out, i, j, carry_in]
    for c_in in (0, 1):
        for j in (0, 1):
            i = (j + c_in) % 2
            c_out = 1 if (j == 1 and c_in == 1) else 0
            core[c_out, i, j, c_in] = 1.0
    cores = [core.copy() for _ in range(n)]
    # TODO: close the LEFT edge so an overflowing carry is kept, not discarded,
    #       and close the right edge with carry_in = 1 as in tn.qtt_shift.
    ...

**Solution** (hidden by default; expand the cell below)

In [ ]:
def qtt_shift_periodic(n):
    core = np.zeros((2, 2, 2, 2))
    for c_in in (0, 1):
        for j in (0, 1):
            i = (j + c_in) % 2
            c_out = 1 if (j == 1 and c_in == 1) else 0
            core[c_out, i, j, c_in] = 1.0
    cores = [core.copy() for _ in range(n)]
    cores[0] = cores[0].sum(axis=0, keepdims=True)   # accept both carry-out states
    cores[-1] = cores[-1][:, :, :, 1:2]              # adding 1
    return cores


P = qtt_shift_periodic(4)
assert np.allclose(tn.mpo_full(P), np.roll(np.eye(16), 1, axis=0))
print("periodic shift ranks:", tn.mpo_ranks(P), "- same rank 2, one different edge")

### 4.7 Beyond finite differences

The QFT has small entanglement and a low-rank MPO (Chen, Stoudenmire & White; Chen &
Lindsey), and so does the Laplace transform (Jaseem *et al.*). García-Ripoll (2021)
tabulates the costs of quantum-inspired MPS algorithms against classical and quantum
ones. Not needed below, but they are what makes spectral methods available in QTT form.

## 5. Heat equation (explicit time stepping)

$$\partial_t u = \alpha\,\partial_x^2 u,\qquad x\in[0,1],\qquad N=2^n,\qquad
u^{k+1} = (\mathbb 1 + \alpha\Delta t\,L)\,u^k .$$

Ingredients: the initial condition $u_0$ as a QTT, the discrete Laplacian $L$ as a rank-3
MPO, and the diffusion number $\mu=\alpha\Delta t/\Delta x^2$. Each step is **two
operations: MPO–MPS product + rounding** $(\chi,\epsilon)$, at cost $O(n\chi^3)$ and
memory $O(n\chi^2)$ — both logarithmic in $N$.

In [ ]:
n, alpha = 16, 1.0
h = 2.0 ** -n
dt = 0.4 * h ** 2 / alpha                     # mu = 0.4 <= 1/2
xs = np.linspace(0, 1, 2 ** n, endpoint=False)

L = tn.qtt_laplacian(n, dx=h)
step = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, alpha * dt)), 1e-13)

u = tn.qtt_from_vector(np.exp(-((xs - 0.5) / 0.05) ** 2), eps=1e-10)
history = [max(tn.tt_ranks(u))]
snapshots = {0: tn.qtt_to_vector(u)}

t0 = time.perf_counter()
for k in range(1, 601):
    u = tn.tt_round(tn.mpo_apply(step, u), eps=1e-8)
    history.append(max(tn.tt_ranks(u)))
    if k in (200, 600):
        snapshots[k] = tn.qtt_to_vector(u)       # dense readout only for plotting
print(f"600 steps on a {2**n:,d}-point grid in {time.perf_counter() - t0:.2f} s")

fig, ax = plt.subplots(1, 2)
for k, vec in snapshots.items():
    ax[0].plot(xs, vec, label=f"step {k}")
ax[0].set(xlim=(0.25, 0.75), xlabel="x", title="diffusion")
ax[0].legend()
ax[1].plot(history)
ax[1].set(xlabel="step", ylabel="max bond dimension", title="$\\chi(t)$")
plt.tight_layout()

**Rank remark.** Diffusion smooths, so $\chi$ should stay flat or fall. If it climbs,
suspect the encoding, the boundary treatment, or too loose a rounding tolerance.

**Stability remark.** Compression does not touch the CFL condition
$\mu = \alpha\Delta t/\Delta x^2 \le 1/2$: the amplification factor of a Fourier mode
with grid angle $\theta$ is $G = 1-4\mu\sin^2(\theta/2)$. With $\Delta x = 2^{-n}$ this
forces $\Delta t\sim 4^{-n}$: the step count grows exponentially in $n$ while the memory
grows linearly. (This is the motivation for implicit schemes, Part II.)

---
### Exercise 5.1 — the CFL bound survives compression


On $n=8$, start from $\sin(\pi x)$ plus a tiny near-Nyquist perturbation
$10^{-6}(-1)^j$ (which is itself rank 1 in QTT — why?). Run 150 rounded steps at
$\mu=0.45$ and at $\mu=0.55$ and plot $\max_j|u_j|$ against the step. **Predict** the
growth factor per step of the perturbation in each case from $|G(\pi)| = |1-4\mu|$.

In [ ]:
n_cfl = 8
h_c = 2.0 ** -n_cfl
x_c = np.linspace(0, 1, 2 ** n_cfl, endpoint=False)
u0_c = np.sin(np.pi * x_c) + 1e-6 * (-1.0) ** np.arange(2 ** n_cfl)

# TODO: for mu in (0.45, 0.55): build (1 + mu * L h^2), step 150 times with
#       tn.tt_round(..., eps=1e-12), record np.abs(tn.qtt_to_vector(u)).max()

**Solution** (hidden by default; expand the cell below)

In [ ]:
L_c = tn.qtt_laplacian(n_cfl, dx=1.0)            # = h^2 * Laplacian
for mu in (0.45, 0.55):
    stp = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n_cfl), tn.mpo_scale(L_c, mu)), 1e-13)
    u = tn.qtt_from_vector(u0_c, eps=1e-14)
    peaks = []
    for _ in range(150):
        u = tn.tt_round(tn.mpo_apply(stp, u), eps=1e-12)
        peaks.append(np.abs(tn.qtt_to_vector(u)).max())
    plt.semilogy(peaks, label=f"$\\mu={mu}$, $|1-4\\mu|={abs(1 - 4 * mu):.1f}$")
plt.xlabel("step"); plt.ylabel("max |u|"); plt.legend()
plt.title("explicit Euler: stable at $\\mu<1/2$, blows up above")
plt.tight_layout()
print("(-1)^j depends only on the last bit x_n: a product state, rank 1.")

---
### Exercise 5.2 — the cost of forgetting to round


**Point:** rounding is the algorithm, not an optimisation. Each MPO application
multiplies the bond dimension by $\chi_{\mathrm{MPO}}=3$, so $\chi_k = 3^k\chi_0$ and the
memory grows like $9^k$. Run explicit Euler **without** rounding and count the steps
before the cores would exceed 256 MiB. Check the *predicted* size of the next state
before computing it — otherwise the cell allocates the thing you were trying to avoid.
Then repeat with `tn.tt_round(..., eps=1e-8)` after each step.

In [ ]:
n_r = 10
h_r = 2.0 ** -n_r
x_r = np.linspace(0, 1, 2 ** n_r, endpoint=False)
L_r = tn.qtt_laplacian(n_r, dx=h_r)
step_r = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n_r), tn.mpo_scale(L_r, 0.4 * h_r ** 2)), 1e-13)
u0_r = tn.qtt_from_vector(np.sin(2 * np.pi * x_r), eps=1e-12)
budget = 256 * 2 ** 20
# TODO: loop without rounding while 9 * tt_size(u) * 8 < budget; print k, chi, 3^k chi_0, MiB

**Solution** (hidden by default; expand the cell below)

In [ ]:
u = [c.copy() for c in u0_r]
chi0 = max(tn.tt_ranks(u))
k = 0
print(f"{'step':>5} {'chi (no rounding)':>18} {'3^k * chi_0':>13} {'memory':>12}")
while 9 * tn.tt_size(u) * 8 < budget:          # next step multiplies memory by ~9
    u = tn.mpo_apply(step_r, u)
    k += 1
    print(f"{k:>5} {max(tn.tt_ranks(u)):>18,d} {3 ** k * chi0:>13,d} "
          f"{tn.tt_size(u) * 8 / 2 ** 20:>10.2f} MiB")
rounded = tn.tt_round(u, eps=1e-10)
print(f"\none rounding of that state: chi = {max(tn.tt_ranks(rounded))} -- "
      "the extra rank was never information, only bookkeeping.")
u = [c.copy() for c in u0_r]
for _ in range(k + 20):
    u = tn.tt_round(tn.mpo_apply(step_r, u), eps=1e-8)
print(f"with rounding, after {k + 20} steps: chi = {max(tn.tt_ranks(u))}, "
      f"{tn.tt_size(u) * 8 / 2 ** 10:.1f} KiB")

### Two sources of error

* **truncation** — QTT solution vs the *same* Euler scheme run densely;
* **discretisation** — dense Euler vs the exact PDE solution.

Below: homogeneous Dirichlet grid $x_j=(j+1)h$, $h=1/(N+1)$, and
$u_0=\sin(\pi x)+\sin(16\pi x)$, whose exact solution is
$e^{-\pi^2 t}\sin(\pi x) + e^{-256\pi^2 t}\sin(16\pi x)$.

In [ ]:
n_e, mu_e, steps_e = 10, 0.4, 2000
N_e = 2 ** n_e
h_e = 1.0 / (N_e + 1)
x_e = np.arange(1, N_e + 1) * h_e
dt_e = mu_e * h_e ** 2
u0_e = np.sin(np.pi * x_e) + np.sin(16 * np.pi * x_e)

step_e = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n_e),
                                 tn.mpo_scale(tn.qtt_laplacian(n_e, dx=1.0), mu_e)), 1e-13)
u_q = tn.qtt_from_vector(u0_e, eps=1e-12)
u_d = u0_e.copy()
for _ in range(steps_e):
    u_q = tn.tt_round(tn.mpo_apply(step_e, u_q), eps=1e-10)
    u_d = u_d + mu_e * (np.r_[u_d[1:], 0.0] + np.r_[0.0, u_d[:-1]] - 2 * u_d)

t_e = steps_e * dt_e
exact = (np.exp(-np.pi ** 2 * t_e) * np.sin(np.pi * x_e)
         + np.exp(-(16 * np.pi) ** 2 * t_e) * np.sin(16 * np.pi * x_e))
rel = lambda a, b: np.linalg.norm(a - b) / np.linalg.norm(b)
print(f"t = {t_e:.2e},  chi = {max(tn.tt_ranks(u_q))}")
print(f"truncation     ||QTT - dense Euler|| / ||dense||  = {rel(tn.qtt_to_vector(u_q), u_d):.2e}")
print(f"discretisation ||dense Euler - exact|| / ||exact|| = {rel(u_d, exact):.2e}")

Know which of the two dominates before you tighten either.

### Going further

* `01-heat-equation-qtmps-quimb.ipynb` — the same explicit scheme in `quimb`, at
  $2^{20}$ points in 1D and $1024^2$ in 2D, with diagnostics computed without dense
  reconstruction, and a comparison of **stacked vs interleaved** ordering in 2D.
* Waintal, Huang & Groth, *Who can compete with quantum computers?* (arXiv:2601.03035),
  pp. 60–61: the heat equation with TCI and an FFT-MPO on $2^{30}$ points.

Next: Part II, where "apply" is no longer enough and we have to *solve*.